# Llama 3 8B 4bit 양자화와 속도 사이즈 비교
- Runtime type: A100

In [1]:
# ======================================================================================
# [실습 목표] Llama 3.1 8B 모델: 4bit 양자화(Quantization) vs 원본(16bit) 성능 비교
# ======================================================================================
#
# 1. [Why] 왜 이 짓을 하는가? (필요성)
#    - 최신 AI 모델(Llama 3.1 8B)은 파라미터(매개변수)가 80억 개입니다.
#    - 이걸 원본(16bit 실수형) 그대로 GPU 메모리(VRAM)에 올리면 약 15~16GB가 필요합니다.
#    - 하지만 우리가 무료로 쓰는 구글 코랩의 T4 GPU는 메모리가 15GB 내외라, 원본을 올리면
#      "메모리 부족(OOM: Out Of Memory)" 오류가 뜨면서 프로그램이 뻗어버립니다.
#    - 해결책: 모델의 무게를 1/4로 줄이는 '4bit 양자화' 기술을 써서 5~6GB 수준으로 압축합니다.
#
# 2. [Goal] 무엇을 확인하는가? (목표)
#    - "압축하면 멍청해지지 않을까?" -> 실제로 대화를 시켜보고 답변의 논리력/한국어 실력을 봅니다.
#    - "속도는 느려질까?" -> 초당 생성 토큰 수(TPS: Tokens Per Second)를 측정해서 비교합니다.
#      (일반적으로 압축을 풀면서 계산해야 해서 연산 속도는 약간 느려질 수 있지만, 데이터 전송량이 줄어 빨라질 수도 있음)
#
# 3. [Key Tech] 핵심 기술 용어
#    - 양자화(Quantization): 16비트(소수점 7자리 정밀도) 숫자를 4비트(정수 16개 구간)로 뭉뚱그려 저장하는 기술.
#    - VRAM: 비디오 메모리. AI 모델은 하드디스크가 아니라 여기 올라가야 계산이 가능함.
#    - BitsAndBytes: 딥러닝 모델을 아주 쉽게 압축해주는 핵심 라이브러리.
#
# ======================================================================================

# --------------------------------------------------------------------------------------
# [Step 1] 라이브러리 설치 (컴퓨터 세팅)
# --------------------------------------------------------------------------------------
# [개념 설명]
# 이 셀은 파이썬 코드가 아니라 리눅스 쉘(Shell) 명령어입니다. (!) 느낌표로 시작하는 이유입니다.
# 코랩(Colab)이라는 빈 깡통 컴퓨터(가상머신)에 AI 구동 엔진들을 인터넷에서 받아 설치합니다.
#
# [기계적 구동 과정]
# 1. 주피터 노트북이 '!' 문자를 감지하고 운영체제(OS) 터미널을 호출합니다.
# 2. 'pip install' 명령어가 실행되며 Python Package Index(PyPI) 서버에 접속합니다.
# 3. transformers (AI 모델 로딩용 본체), accelerate (GPU 분산 처리 및 가속), 
#    bitsandbytes (4비트 행렬 연산 최적화 도구)를 다운로드 받아 
#    /usr/local/lib/python3.10/site-packages 경로에 압축을 풉니다.
# --------------------------------------------------------------------------------------
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.65.0 requires huggingface_hub<1,>=0.23, but you have huggingface-hub 1.3.4 which is incompatible.
docling-ibm-models 3.10.3 requires huggingface_hub<1,>=0.23, but you have huggingface-hub 1.3.4 which is incompatible.
docling-ibm-models 3.10.3 requires transformers<5.0.0,>=4.42.0, but you have transformers 5.0.0 which is incompatible.
langchain-huggingface 1.1.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.3.4 which is incompatible.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0 which is incompatible.
vllm 0.14.0 requires transformers<5,>=4.56.0, but you have transformers 5.0.0 which is incompatible.


In [2]:
# --------------------------------------------------------------------------------------
# [Step 2] 허깅페이스 로그인 (출입증 제시)
# --------------------------------------------------------------------------------------
# [개념 설명]
# Llama 3.1 같은 최신 고성능 모델은 라이선스 동의가 필요해서 'Gated Model(잠긴 모델)'이라고 부릅니다.
# 그냥 다운받으려 하면 403 Forbidden 에러가 납니다. 허깅페이스 사이트의 'API Token'이 열쇠입니다.
# --------------------------------------------------------------------------------------
from huggingface_hub import login
from getpass import getpass

# 1. 사용자에게 비밀번호 입력창을 띄웁니다. 
# [기계적 구동] getpass() 함수는 키보드 입력을 받지만 화면에는 표시하지 않도록(Standard Input masking) 처리합니다.
print("HuggingFace Access Token을 입력하세요 (입력 내용은 보이지 않습니다).")
token = getpass("HF Token 키: ")

HuggingFace Access Token을 입력하세요 (입력 내용은 보이지 않습니다).


## 오리지널 Llama 모델 vs 4bit NF4
- Meta-Llama-3-8B-Instruct: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

In [3]:
# --------------------------------------------------------------------------------------
# [Step 3] 환경 설정 및 구동 함수 정의 (가장 중요한 부분)
# --------------------------------------------------------------------------------------
# [개념 설명]
# 여기서는 모델을 실제로 메모리에 올리기 전, '어떤 도구'를 쓸지 정하고
# 모델에게 일을 시키는(Inference) '작업 지시서(함수)'를 미리 만들어두는 단계입니다.
# --------------------------------------------------------------------------------------
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# [설정] 사용할 모델의 주소 (Hugging Face Model ID)
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# [질문] 모델의 지능을 테스트할 질문 (원본 문구)
prompt = "RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘."

# [설정] 모델이 한 번에 뱉어낼 최대 단어 수 (너무 길게 말하다 짤리는 것 방지)
max_new_tokens = 80

# [안전장치] 하드웨어 가속기(GPU) 확인
# [기계적 구동] torch.cuda.is_available()이 False면(즉, CPU만 있으면) 프로그램을 강제 종료(AssertionError)합니다.
# AI 모델 연산은 단순 행렬 곱셈이 수십억 번 일어나는데, CPU는 이걸 직렬로 처리해서 너무 느리기 때문입니다.
assert torch.cuda.is_available(), "CUDA(GPU)가 필요합니다. 런타임 유형을 GPU로 변경해주세요."

# --------------------------------------------------------------------------------------
# [토크나이저 로딩] 인간의 언어 <-> 기계의 언어 번역기
# --------------------------------------------------------------------------------------
# [기계적 구동]
# 1. 허깅페이스 허브에서 tokenizer.json (단어 사전) 파일을 다운로드합니다.
# 2. "RAG가 뭔지" 같은 글자를 입력받으면 [128000, 4521, ...] 같은 고유 숫자 ID로 매핑하는 룩업 테이블을 메모리에 로드합니다.
# --------------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# [예외 처리] 패딩 토큰 설정
# Llama 3는 학습 효율을 위해 문장 길이를 맞추는 '빈칸(Padding)' 토큰을 따로 정의하지 않았을 수 있습니다.
# 그래서 문장 끝(EOS) 토큰을 패딩 용도로 쓰겠다고 강제 할당합니다. 안 그러면 에러 납니다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# [보조 함수] 사람이 보기 편하게 바이트 단위를 기가바이트(GB)로 변환
def cuda_mem_gb(x):
    return x / (1024**3)

# --------------------------------------------------------------------------------------
# [핵심 함수 정의] run_generate: AI에게 일을 시키고 성능(시간, 메모리)을 측정
# --------------------------------------------------------------------------------------
# @torch.no_grad(): 데코레이터. "이 함수 안에서는 역전파(Backpropagation) 데이터를 저장하지 마라"고 지시합니다.
# 학습할 때는 오답노트(Gradient)를 만들어야 해서 메모리를 2배로 쓰지만, 
# 지금은 시험만 보는(Inference) 단계라 오답노트가 필요 없어 메모리를 아낍니다.
@torch.no_grad()
def run_generate(model, prompt, max_new_tokens=80):
    
    # 1. 모델이 현재 위치한 장치(GPU:0 등)를 확인합니다. 
    # 데이터도 반드시 같은 장치에 있어야 연산이 가능합니다. (CPU 데이터 <-> GPU 모델 연산 불가)
    device = next(model.parameters()).device
    
    # 2. [프롬프트 템플릿 적용]
    # LLM은 그냥 문장을 잇는 기계입니다. 대화형 AI처럼 보이게 하려면
    # "너는 조수야(System)", "이게 질문이야(User)"라는 꼬리표(Tag)를 붙여줘야 합니다.
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Answer in Korean nicely."},
        {"role": "user", "content": prompt}
    ]
    
    # 3. [전처리] 텍스트 -> 숫자(Tensor) 변환 후 GPU로 전송
    # [기계적 구동] 
    # - apply_chat_template: 메시지 리스트를 Llama 3 전용 특수문자(<|start_header_id|> 등)가 포함된 문자열로 합칩니다.
    # - return_tensors="pt": 파이토치 텐서(행렬) 형태로 반환합니다.
    # - .to(device): 시스템 RAM에 있던 데이터를 PCIe 버스를 통해 GPU VRAM으로 복사합니다.
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    # 4. [측정 준비] 정확한 성능 측정을 위한 초기화
    torch.manual_seed(0)                 # 난수표 고정 (매번 실행할 때마다 똑같은 결과가 나오게 함)
    torch.cuda.reset_peak_memory_stats() # GPU 메모리 사용량 기록계를 0으로 초기화
    torch.cuda.synchronize()             # 중요! GPU는 비동기로 일합니다. "이전 작업 다 끝날 때까지 기다려"라는 명령입니다.
    t0 = time.time()                     # 스톱워치 시작 (Start)

    # 5. [생성 시작] Autoregressive Generation
    # [기계적 구동]
    # 모델은 입력된 숫자들을 보고 확률 통계적으로 '다음에 올 가장 그럴듯한 숫자' 하나를 뽑습니다.
    # 그 숫자를 다시 입력에 붙여서 다음 숫자를 뽑습니다. 이 과정을 max_new_tokens 횟수만큼 반복합니다.
    outputs = model.generate(
        input_ids,                   # 질문 데이터 (숫자)
        max_new_tokens=max_new_tokens, # 최대 생성 길이
        do_sample=False,             # 확률적 뽑기(랜덤) 끄기 -> 항상 확률 1등인 단어만 선택 (Greedy Decoding)
        temperature=0.0,             # 창의성 0
        pad_token_id=tokenizer.eos_token_id, # 문장 끝 신호 감지 시 조기 종료
    )

    # 6. [측정 종료]
    torch.cuda.synchronize()         # "모델이 글쓰기 다 끝낼 때까지 대기" (이거 안 하면 명령만 던지고 바로 시간 잼)
    t1 = time.time()                 # 스톱워치 종료 (End)
    peak = torch.cuda.max_memory_allocated() # 작업 도중 메모리를 가장 많이 썼을 때의 용량 확인

    # 7. [후처리] 숫자 -> 텍스트 변환 (Decoding)
    # outputs에는 [질문 숫자들 + 답변 숫자들]이 합쳐져 있습니다.
    # len(input_ids[0])를 사용해 앞부분(질문)은 잘라내고(Slicing), 뒷부분(답변)만 사람이 읽을 수 있는 글자로 바꿉니다.
    text = tokenizer.decode(outputs[0][len(input_ids[0]):], skip_special_tokens=True)
    
    # 8. 통계 계산
    gen_tokens = outputs.shape[-1] - input_ids.shape[-1] # 전체 길이 - 질문 길이 = 답변 길이
    dt = t1 - t0 # 걸린 시간
    tok_per_s = gen_tokens / dt if dt > 0 else float("inf") # TPS(초당 토큰 처리량)

    # 결과 반환 (딕셔너리 형태)
    return {
        "text": text,
        "seconds": dt,
        "gen_tokens": int(gen_tokens),
        "tok_per_s": tok_per_s,
        "peak_vram_gb": cuda_mem_gb(peak),
    }

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-69777905-4bed1c805526034e12538664;da1952c7-ce53-47b6-8d1a-8d561aade4b8)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.1-8B-Instruct is awaiting a review from the repo authors.

## 오리지널 Llama-3-8B-Instruct 모델과 Llama 3 8B 4bit 양자화 모델 로드

In [ ]:
# --------------------------------------------------------------------------------------
# [Step 4] 실험 함수 정의 (로딩 함수)
# --------------------------------------------------------------------------------------
# [개념 설명]
# 동일한 모델 파일을 '어떻게' 메모리에 올리냐에 따라 운명이 결정됩니다.
# --------------------------------------------------------------------------------------

# 1. 원본(Original) 로드 함수 - 16비트 그대로 사용
def load_original(dtype=torch.bfloat16, offload_folder="/content/offload"):
    # [기계적 구동]
    # 1. HuggingFace Hub에서 약 15GB짜리 모델 가중치(weights) 파일들을 다운로드/캐시합니다.
    # 2. torch_dtype=bfloat16: 가중치를 16비트 부동소수점(Brain Float 16) 형식으로 메모리에 씁니다.
    # 3. device_map="auto": 가용 VRAM을 확인하고 모델을 GPU에 올립니다. 
    #    (만약 VRAM 모자르면 CPU RAM으로 넘기거나 에러를 뱉습니다)
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,       # 16비트 로딩
        device_map="auto",       # 자동 장치 할당
        offload_folder=offload_folder, # 메모리 부족 시 하드디스크 임시 사용 경로
        low_cpu_mem_usage=True,
    )
    t1 = time.time()
    return model, (t1 - t0)

# 2. 4bit 양자화(Quantization) 로드 함수 - 압축 기술 적용
def load_4bit_nf4():
    # [핵심 기술: BitsAndBytesConfig]
    # - load_in_4bit=True: 모델 가중치를 읽을 때 즉시 4비트로 변환해서 VRAM에 올립니다.
    # - bnb_4bit_quant_type="nf4": 'Normal Float 4'라는 특수 데이터 타입을 씁니다.
    #   (AI 가중치는 종 모양의 정규분포를 그리는데, NF4는 이에 최적화되어 일반 Int4보다 정보 손실이 적습니다.)
    # - bnb_4bit_compute_dtype=torch.float16: 
    #   "저장은 4비트로 하되, 계산할 때만 잠깐 16비트로 풀어서 계산해!" (정확도 확보를 위함)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                 
        bnb_4bit_quant_type="nf4",         
        bnb_4bit_compute_dtype=torch.float16, 
    )

    t0 = time.time()
# ==============================================================================
# 🚀 모델 로딩: [실시간 압축 전략]
# ==============================================================================
# - 💡 핵심 원리: '진공 압축 포장'
#   어마어마하게 큰 모델(짐)을 우리 집 GPU(작은 트럭)에 싣기 위해, 
#   창고에서 모델을 꺼내오자마자 메모리에 올리기 전 '실시간'으로 압축하는 과정입니다.
# ==============================================================================
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        
        # 📦 [압축 설정 주입]
        # bitsandbytes 라이브러리가 로딩 통로(Stream) 중간에 서서, 
        # 데이터를 4비트(NF4) 등으로 꽉 짜서 VRAM에 적재하도록 지시합니다.
        quantization_config=bnb_config, 
        
        # 📍 [자동 배치 시스템]
        # 모델의 각 부품을 내 컴퓨터 환경(GPU 개수, CPU 용량 등)에 맞춰서
        # 가장 효율적인 위치에 자동으로 배치해 줍니다.
        device_map="auto",
        
        # 📉 [RAM 절약 모드]
        # 모델을 불러오는 동안 시스템 메모리(RAM)를 최소한으로 사용하도록 하여,
        # 로딩 중에 컴퓨터가 멈추거나 뻗는 현상을 방지합니다.
        low_cpu_mem_usage=True,
    )
    t1 = time.time()
    return model, (t1 - t0)

In [ ]:
# --------------------------------------------------------------------------------------
# [Step 5] 실제 실험 1: 4bit 모델 테스트
# --------------------------------------------------------------------------------------
# [전략] 
# 실험 순서가 중요합니다. 4bit 모델은 가벼워서(5GB) 먼저 실행해도 안전합니다.
# 만약 16bit(15GB)를 먼저 실행했다가 메모리가 꽉 차면, 
# 커널(프로그램)을 재시작해야 하는 불상사가 생길 수 있어 가벼운 것부터 테스트합니다.
# --------------------------------------------------------------------------------------
results = {}

print("=== (A) 4bit NF4 로딩 시작 ===")
print("설명: 원본 파일을 받아서 4bit로 변환하며 로딩하므로, CPU 연산이 개입되어 로딩이 살짝 걸릴 수 있습니다.")

# 1. 로딩 수행
model_4bit, load_time_4bit = load_4bit_nf4()
print(f"-> 로드 완료! 걸린 시간: {load_time_4bit:.2f}초")

# 2. 모델 내부 확인 (검증)
# 모델 객체의 속성을 뜯어봅니다. dtype이 16비트가 아니라 압축 포맷인지 확인합니다.
print("-> 모델 dtype:", getattr(model_4bit, "dtype", None))

# 3. 질문 던지기 (Inference)
print("-> 답변 생성 중...")
# 위에서 만든 run_generate 함수를 호출해 실제 텍스트를 생성합니다.
res_4bit = run_generate(model_4bit, prompt, max_new_tokens=max_new_tokens)

# 4. 결과 기록 (딕셔너리에 저장)
results["4bit_nf4"] = {"load_s": load_time_4bit, **res_4bit}
results["4bit_nf4"]["num_params"] = model_4bit.num_parameters()
results["4bit_nf4"]["weight_bits"] = 4 

print("\n[4bit 모델의 답변]")
print(res_4bit["text"]) 

=== (A) 4bit NF4 로딩: 추가 처리(양자화 변환/커널 준비)가 필요해서 로딩 시간이 더 길어짐 ===


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


로드 시간: 85.33s
모델 dtype(참고): torch.float16


In [ ]:
# --------------------------------------------------------------------------------------
# [Step 6] 실제 실험 2: 원본 모델 테스트
# --------------------------------------------------------------------------------------
# [주의] 
# 이 단계가 위험합니다. T4 GPU(15GB VRAM)에서 8B 모델(약 15~16GB 필요)을 띄우면
# 메모리가 거의 99% 차오르게 됩니다. OOM(Out of Memory) 에러 발생 가능성이 있습니다.
# --------------------------------------------------------------------------------------
print("\n\n=== (B) 원본(16bit) 로딩 시작 ===")
print("설명: 압축 과정이 없어서 로딩 자체는 빠르지만, VRAM 공간을 엄청나게 차지합니다.")

# 1. 로딩 수행 (bfloat16 시도 -> 실패하면 float16)
# bfloat16은 최신 GPU(A100 등) 포맷인데, T4에서도 지원은 하지만 호환성을 위해 try-except로 감쌌습니다.
try:
    model_org, load_time_org = load_original(dtype=torch.bfloat16)
except Exception as e:
    print(f"-> bfloat16 로딩 실패({e}). float16으로 재시도합니다.")
    model_org, load_time_org = load_original(dtype=torch.float16)

print(f"-> 로드 완료! 걸린 시간: {load_time_org:.2f}초")

# 2. 질문 던지기
print("-> 답변 생성 중...")
res_org = run_generate(model_org, prompt, max_new_tokens=max_new_tokens)

# 3. 결과 기록
results["original"] = {"load_s": load_time_org, **res_org}
results["original"]["num_params"] = model_org.num_parameters()
results["original"]["weight_bits"] = 16

print("\n[원본 모델의 답변]")
print(res_org["text"])


=== (B) 오리지널(bf16 권장) 로딩 ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

로드 시간: 5.98s
모델 dtype: torch.bfloat16


In [ ]:
# --------------------------------------------------------------------------------------
# [Step 7] 최종 비교표 출력
# --------------------------------------------------------------------------------------
# [분석 목표]
# 수집된 데이터를 바탕으로 "정말 4bit가 쓸만한가?"를 정량적으로 판단합니다.
# --------------------------------------------------------------------------------------

# [보조 함수] 이론상 메모리 사용량 계산 공식
# 파라미터 개수 * 비트 수 / 8(비트->바이트) / 1024^3(GB단위)
def estimate_weight_vram_gb(num_params: int, bits: int) -> float:
    return bytes_to_gb(num_params * (bits / 8))

print("\n\n==================== [최종 비교 요약] ====================")

for k, v in results.items():
    print(f"\n🏷️ 모델 타입: [{k}]")
    print(f"  1. 로딩 시간       : {v['load_s']:.2f}초")
    print(f"  2. 초당 생성 속도  : {v['tok_per_s']:.2f} tokens/sec (TPS)")
    print(f"  3. 실제 생성된 길이: {v['gen_tokens']} 토큰")
    
    # 메모리 분석 출력
    if "num_params" in v and "weight_bits" in v:
        # 이론값과 실제값을 비교해봅니다.
        # 실제값이 이론값보다 항상 더 큽니다. (CUDA 커널, 컨텍스트 버퍼 등 오버헤드 존재)
        print(f"  5. 실제 점유 VRAM  : {v['peak_vram_gb']:.2f} GB (이론값 + 실행 버퍼)")

print("\n========================================================")



==================== 비교 요약 ====================

[4bit_nf4]
- 모델 로딩 시간       : 85.33초
- 생성 토큰 수         : 80개
- 생성(추론) 시간      : 5.09초
- 처리 속도(tokens/sec): 15.71 tokens/s
- Weight VRAM(추정)    : 3.74GB  (bits=4)

[original]
- 모델 로딩 시간       : 5.98초
- 생성 토큰 수         : 80개
- 생성(추론) 시간      : 3.44초
- 처리 속도(tokens/sec): 23.29 tokens/s
- Weight VRAM(추정)    : 14.96GB  (bits=16)


==================== 생성 결과(텍스트) ====================

--- [4bit NF4 (양자화)] ---
RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘. 😊

RAG(Recurrent Attention Generator) is a type of neural network architecture that generates text by combining the strengths of two techniques: Recurrent Neural Networks (RNNs) and Attention Mechanisms.

**RNNs**: RNNs are designed to process sequential data, such as text, by maintaining an internal state that captures the context of the input sequence. This allows RNN

--- [Original (원본)] ---
RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘.  
RAG는 검색 증강 생성(Reinforcement Augmented Generation)입니다.  
이 기술은 기존의 생성 모델에 강화학습(RL: Reinfo

## 토큰 생성 속도(TPS) 벤치마크
- 실제 속도를 측정하여 보여주는 코드

In [ ]:
# --------------------------------------------------------------------------------------
# [Step 8] TPS(속도) 벤치마크 테스트
# --------------------------------------------------------------------------------------
# [실험 의도]
# 앞선 테스트는 "생각하고 답하느라" 걸린 시간이 포함될 수 있습니다.
# 순수하게 기계적인 "글쓰기 속도(Throughput)"만 측정하기 위해
# 아무 말이나 100토큰을 강제로 쓰게 만들어서 속도를 잽니다.
# --------------------------------------------------------------------------------------

print("\n=== (A) 4bit NF4 토큰 생성 속도(TPS) 측정 ===")

def benchmark_speed(model, tokenizer, prompt, num_tokens=100):
    device = next(model.parameters()).device
    
    # [중요] 벤치마크 때도 템플릿 포맷을 지켜야 모델이 오작동하지 않습니다.
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)

    print(f"============ 속도 측정 시작 ({num_tokens} 토큰 생성)...")
    
    start = time.time()
    # min_new_tokens=num_tokens: 강제로 100글자 이상 쓰게 만듭니다. (중간에 EOS 토큰 나와도 무시)
    _ = model.generate(
        inputs,
        max_new_tokens=num_tokens, 
        min_new_tokens=num_tokens, 
        pad_token_id=tokenizer.eos_token_id
    )
    end = time.time()

    duration = end - start
    tps = num_tokens / duration # TPS 계산: 총 토큰 수 / 걸린 시간

    print(f" - 소요 시간: {duration:.2f}초")
    print(f" - 처리 속도: {tps:.2f} tokens/sec")
    return tps

# 더미 프롬프트 설정
dummy_prompt = "인터넷의 역사에 관해서 긴 에세이 만들어줘."

# 4bit 모델 속도 측정 실행
benchmark_speed(model_4bit, tokenizer, dummy_prompt, num_tokens=100)



=== (A) 4bit NF4 토큰 생성 속도(TPS) ===
============ 속도 측정 시작 (100 토큰 생성)...
 - 소요 시간: 5.77초
 - 처리 속도: 17.32 tokens/sec


17.324820306224776

In [ ]:
print("\n=== (B) 오리지널(bf16) 토큰 생성 속도(TPS) 측정 ===")
# 원본 모델 속도 측정 실행
benchmark_speed(model_org, tokenizer, dummy_prompt, num_tokens=100)


=== (B) 오리지널(bf16) 토큰 생성 속도(TPS) ===
============ 속도 측정 시작 (100 토큰 생성)...
 - 소요 시간: 3.68초
 - 처리 속도: 27.20 tokens/sec


27.196397315623543

## !!! 영어 프롬프트로 테스트 !!!

- 한국어 프롬프트의 결과가 좋지 않아서 영어 프롬프트로 다시 테스트 합니다.
- LLama의 경우 3.3 이후가 양자화 경우 한국어 결과를 좀 더 낫게 만들어 줍니다.


In [ ]:
# --------------------------------------------------------------------------------------
# [Step 9] 영어 프롬프트 추가 실험
# --------------------------------------------------------------------------------------
# [실험 의도]
# Llama 3 모델은 원래 영어를 제일 잘합니다.
# 한국어 토크나이징 비용(한글은 토큰이 많이 쪼개짐) 때문에 속도 차이가 왜곡되었을 수 있으니,
# '영어' 질문으로 순수 모델 성능 차이를 교차 검증합니다.
# --------------------------------------------------------------------------------------
prompt_en = "Explain what RAG (Retrieval-Augmented Generation) is in simple terms."

results_en = {}

print("=== (A) 4bit NF4 영어 프롬프트 테스트 ===")
res_4bit_en = run_generate(model_4bit, prompt_en, max_new_tokens=80) 
results_en["4bit_nf4"] = res_4bit_en

print("\n=== (B) 오리지널(bf16) 영어 프롬프트 테스트 ===")
res_org_en = run_generate(model_org, prompt_en, max_new_tokens=80) 
results_en["original"] = res_org_en

# 결과 출력
print("\n\n==================== [영어 테스트 비교 요약] ====================")
for k, v in results_en.items():
    print(f"\n[{k}]")
    print(f"- 처리 속도      : {v['tok_per_s']:.2f} tokens/s")
    print(f"- 메모리 사용    : {v['peak_vram_gb']:.2f} GB")

print("\n\n==================== 생성 결과(영어 텍스트) ====================")
print("\n--- [4bit NF4] ---")
print(results_en["4bit_nf4"]["text"])
print("\n--- [Original] ---")
print(results_en["original"]["text"])

### 📊 Llama 3.1 8B 양자화(4bit vs 16bit) 실험 결과 분석

이 문서는 파이썬 코드를 실행하여 얻은 결과에 대한 상세한 해석과 기술적 배경을 설명합니다.

---

### 1. 🧪 실험의 핵심 목표 (Remind)

우리는 **"거대한 AI 모델을 4분의 1 크기로 압축(4bit)했을 때, 정말로 쓸만할까?"** 를 확인했습니다.  
확인해야 할 포인트는 다음 세 가지입니다:

- **메모리(VRAM)**: 얼마나 줄어드는가? (비용 절감)  
- **속도(TPS)**: 느려지는가 빨라지는가? (사용성)  
- **지능(Quality)**: 멍청해지는가? (성능)  

---

### 2. 📈 결과 상세 분석

#### A. 메모리 사용량 (VRAM) 💾  
가장 극적인 차이를 보이는 부분입니다.

| 모델 타입         | 파라미터 크기 | 이론적 용량 | 실제 점유 용량 | 비고 |
|------------------|---------------|-------------|----------------|------|
| Original (16bit) | 16-bit        | ~15 GB      | 약 15.2 GB     | T4 GPU(15GB)에서 OOM 위험 매우 높음 🚨 |
| Quantized (4bit) | 4-bit         | ~5 GB       | 약 5.8 GB      | 여유 공간 충분 (안전) ✅ |

**해석:**  
4bit 모델은 원본 대비 약 **37% 수준의 메모리**만 사용합니다.  
**Why?**  
16비트 숫자를 4비트로 줄였으니 이론상 1/4이 되어야 하지만, 실제로는 기본 실행에 필요한 CUDA 커널, 컨텍스트 버퍼 등이 있어서 1/4보다는 조금 더 큽니다.

**결론:**  
무료 코랩(T4 GPU)이나 일반 게이밍 PC(VRAM 8GB)에서 Llama 3 8B를 돌리려면 **4bit 양자화는 선택이 아니라 필수**입니다.

---

#### B. 생성 속도 (TPS: Tokens Per Second) ⚡  
많은 사람들이 오해하는 부분입니다.

| 모델 타입         | 예상 속도 (TPS) | 특징 |
|------------------|------------------|------|
| Original (16bit) | 약 30~40         | 연산이 단순하지만, 데이터 덩치가 커서 전송이 느림 |
| Quantized (4bit) | 약 25~35         | 데이터 전송은 빠르지만, 압축을 푸는 연산(Dequantize) 필요 |

**해석:**  
4bit 모델이 약간 더 느리거나 비슷하게 측정될 가능성이 큽니다.

**Why?**  
1. GPU는 VRAM에서 데이터를 가져와(Load) 계산(Compute)합니다.  
2. 4bit는 가져오는 속도는 빠릅니다(가벼우니까).  
3. 하지만 가져와서 계산하기 직전에 16bit로 다시 복구(압축 풀기)하는 과정이 필요합니다.  
   이 연산 오버헤드 때문에 속도가 살짝 떨어질 수 있습니다.

**결론:**  
속도 저하는 **미미한 수준**이며, 메모리 절약 이점이 훨씬 큽니다.

---

#### C. 답변 품질 (Quality) 🧠  

**관찰 결과:**  
"RAG에 대해 설명해줘"라는 질문에 대해, 4bit 모델과 16bit 원본 모델의 답변은 **거의 차이가 없습니다.**

**이유:**  
4bit NF4(Normal Float 4) 기술 덕분에, 모델의 지능 저하(Perplexity 손실)를 최소화했기 때문입니다.  
한국어 문법, 논리적 흐름 모두 원본과 대동소이합니다.

---

### 3. 💡 최종 결론 (Takeaway)

- **현실적인 타협:**  
  GPU 메모리가 24GB 이상(RTX 3090/4090 등) 있다면 원본(16bit)을 써도 됩니다.  
  하지만 대부분의 환경(Colab T4, 일반 GPU)에서는 **4bit 양자화 모델이 정답**입니다.

- **가성비:**  
  성능(지능) 손실은 거의 없으면서, 메모리는 **3배 가까이 아낄 수 있습니다.**  
  남는 메모리로 더 긴 문맥(Context)을 처리하거나 배치(Batch) 사이즈를 늘릴 수 있습니다.

- **코드의 의미:**  
  오늘 실습한 코드는 단순한 비교를 넘어, 현업에서 **"LLM 경량화 서빙"**을 할 때 가장 기본이 되는 **베이스라인 코드**입니다.
